# Model Validation Toolkit — Demo & Usage Guide

**Propósito:** Este notebook demuestra el uso estándar de `validation-suite` para el equipo de validación. Cada sección corresponde a un tipo de prueba definido según los steps definidos en .

**Secciones:**
1. Instalación y configuración
2. `compare_dataframes` — Benchmarking / Dry Run vs Model Owner
3. `vif_check` — Test de multicolinealidad (supuestos de regresión)
4. Exportar evidencia a Excel (trazabilidad para auditoría)
5. Flujo completo de validación

---
> **Nota:** Reemplaza los DataFrames sintéticos por los datos reales del modelo bajo revisión. La estructura de inputs y la interpretación de outputs es idéntica.

## 1. Instalación y configuración

Instalar desde PyPI (una sola vez por ambiente):

In [ ]:
# Instalar la librería
# !pip install validation-suite

# Para desarrollo local (desde el repo clonado):
# !pip install -e "C:/Users/usuario/uvRepos/validation-suite"

In [1]:
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

# Importar funciones de la librería
from validation_suite import (
    compare_dataframes,
    vif_check,
    ValidationResult,
)

# Configuración de display
pd.set_option('display.float_format', '{:.6f}'.format)
pd.set_option('display.max_columns', 20)

print("Librería cargada correctamente.")

Librería cargada correctamente.


---
## 2. `compare_dataframes` — Dry Run vs Model Owner

### Contexto SR 11-7
La sección **IV.A** del SR 11-7 requiere que el validador replique de forma independiente los outputs del model owner. `compare_dataframes` estandariza esa comparación: verifica schema, alinea por llave primaria, y cuantifica diferencias numéricas con tolerancias configurables.

**Cuándo usarla:**
- Al comparar tu corrida independiente (dry run) contra los outputs oficiales del dueño del modelo
- Al verificar que una nueva versión del modelo produce los mismos outputs que la versión en producción
- Al comparar outputs de dos entornos distintos (dev vs prod)

### 2.1 Datos de ejemplo — Modelo PD (Probability of Default)

In [ ]:
# ── REEMPLAZAR CON DATOS REALES ─────────────────────────────────────────────
# df_model_owner  = pd.read_csv('path/to/model_owner_output.csv')
# df_dry_run      = pd.read_csv('path/to/validator_dryrun.csv')
# ────────────────────────────────────────────────────────────────────────────

# Datos sintéticos representativos de un modelo PD
rng = np.random.default_rng(42)
n = 500

obligor_ids = [f"OBL-{i:05d}" for i in range(1, n + 1)]
pd_scores   = rng.beta(a=2, b=18, size=n)          # distribucion tipica de PD retail
lgd_values  = rng.beta(a=4, b=6,  size=n)          # LGD entre 0.2 y 0.8
ead_values  = rng.lognormal(mean=11.5, sigma=1.2, size=n)  # EAD en USD

df_model_owner = pd.DataFrame({
    'obligor_id': obligor_ids,
    'pd_score':   pd_scores,
    'lgd':        lgd_values,
    'ead':        ead_values,
    'segment':    rng.choice(['Retail', 'SME', 'Corporate'], size=n),
})

# Simular dry run del validador: 
# - Mayoría de filas idénticas
# - 8 obligors con diferencias numéricas pequeñas (ej: diferencia de implementación)
df_dry_run = df_model_owner.copy()
idx_diff = rng.choice(n, size=8, replace=False)
df_dry_run.loc[idx_diff, 'pd_score'] += rng.uniform(0.001, 0.008, size=8)

print(f"Model Owner rows : {len(df_model_owner):,}")
print(f"Dry Run rows     : {len(df_dry_run):,}")
print(f"Rows with delta  : {len(idx_diff)}")
df_model_owner.head()

### 2.2 Caso 1 — Comparación con tolerancia estricta (default)

In [ ]:
result_strict = compare_dataframes(
    df_reference=df_model_owner,
    df_challenger=df_dry_run,
    key_cols=['obligor_id'],
    numeric_tol=1e-6,                        # tolerancia estricta (default)
    label_reference='Model Owner v2.3',
    label_challenger='Validator Dry Run',
)

print(f"Status   : {result_strict.status}")
print(f"Warnings : {result_strict.warnings if result_strict.warnings else 'None'}")
print()
result_strict.summary_df.sort_values('max_abs_diff', ascending=False)

**Interpretación:** Las columnas con `rows_exceeding_tol > 0` requieren análisis adicional. Una diferencia numérica pequeña (ej. `max_abs_diff < 0.005`) puede ser aceptable si se explica por diferencias de redondeo entre plataformas — documentar en el reporte de validación.

### 2.3 Caso 2 — Tolerancia permisiva (diferencias de redondeo aceptables)

In [ ]:
result_permissive = compare_dataframes(
    df_reference=df_model_owner,
    df_challenger=df_dry_run,
    key_cols=['obligor_id'],
    numeric_tol=0.01,                        # tolerancia del 1% — diferencias de redondeo
    label_reference='Model Owner v2.3',
    label_challenger='Validator Dry Run',
)

print(f"Status con tol=1e-6 : {result_strict.status}")
print(f"Status con tol=0.01 : {result_permissive.status}")
print()
print("Diferencias que pasan con tol=0.01:")
result_permissive.summary_df[['column', 'max_abs_diff', 'rows_exceeding_tol']]

### 2.4 Caso 3 — Schema mismatch (columnas faltantes)

In [ ]:
# Simular entrega incompleta del Model Owner (falta columna EAD)
df_incomplete = df_dry_run.drop(columns=['ead'])

result_schema = compare_dataframes(
    df_reference=df_model_owner,
    df_challenger=df_incomplete,
    key_cols=['obligor_id'],
)

print("Schema warnings detectados:")
for w in result_schema.warnings:
    print(f"  ⚠  {w}")

print(f"\nStatus: {result_schema.status}")
print("\nColumnas que pudieron compararse:")
result_schema.summary_df[['column', 'max_abs_diff', 'rows_exceeding_tol']]

---
## 3. `vif_check` — Test de Multicolinealidad

### Contexto SR 11-7
Se exige validar los **supuestos del modelo**. Para regresiones (OLS, logística), la multicolinealidad entre predictores infla los errores estándar y hace los coeficientes inestables. El VIF (Variance Inflation Factor) es el test estándar para detectarla.

**Umbrales de referencia SR 11-7:**
| VIF | Clasificación | Acción recomendada |
|-----|--------------|-------------------|
| < 5 | OK | Sin acción |
| 5 – 10 | MODERATE | Monitorear, documentar justificación |
| > 10 | HIGH | Requiere remediación o justificación fuerte |

**Cuándo usarla:**
- En todo modelo de regresión OLS o logística
- Al agregar nuevas variables a un modelo existente
- Como parte de sensitivity analysis en modelos de crédito (PD, LGD)

### 3.1 Caso 1 — Modelo bien especificado (features independientes)

In [ ]:
# ── REEMPLAZAR CON DATOS REALES ─────────────────────────────────────────────
# df_features = pd.read_csv('path/to/model_features.csv')
# feature_cols = ['ltv', 'dti', 'credit_age', 'utilization_rate', 'delinquencies']
# ────────────────────────────────────────────────────────────────────────────

rng = np.random.default_rng(7)
n_obs = 1000

# Features con correlaciones bajas (modelo bien especificado)
df_features_clean = pd.DataFrame({
    'ltv':              rng.uniform(0.30, 0.95, n_obs),
    'dti':              rng.uniform(0.10, 0.60, n_obs),
    'credit_age_yrs':   rng.uniform(1,    30,   n_obs),
    'utilization_rate': rng.uniform(0,    1,    n_obs),
    'num_delinquencies':rng.poisson(lam=0.4, size=n_obs).astype(float),
})

FEATURE_COLS = ['ltv', 'dti', 'credit_age_yrs', 'utilization_rate', 'num_delinquencies']

result_vif_clean = vif_check(
    df=df_features_clean,
    feature_cols=FEATURE_COLS,
    threshold=10.0,          # umbral SR 11-7 estándar
)

print(f"Status: {result_vif_clean.status}")
print()
result_vif_clean.summary_df.sort_values('VIF', ascending=False)

### 3.2 Caso 2 — Multicolinealidad alta (feature redundante)

In [ ]:
# Escenario común: el model owner incluyó tanto el ingreso mensual
# como el ingreso anualizado — son la misma variable escalada
df_features_collinear = df_features_clean.copy()
noise = rng.normal(0, 0.001, n_obs)
df_features_collinear['dti_annualized'] = df_features_collinear['dti'] * 12 + noise

FEATURES_COLLINEAR = FEATURE_COLS + ['dti_annualized']

result_vif_collinear = vif_check(
    df=df_features_collinear,
    feature_cols=FEATURES_COLLINEAR,
    threshold=10.0,
)

print(f"Status: {result_vif_collinear.status}")
print()
print("Features problemáticas:")
for f in result_vif_collinear.details['flagged_features']:
    print(f"  ► {f}")
print()
result_vif_collinear.summary_df.sort_values('VIF', ascending=False)

### 3.3 Visualización del resumen VIF

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

def plot_vif(result: ValidationResult, title: str = "VIF por variable") -> None:
    """Bar chart de VIF con bandas de umbral SR 11-7."""
    df = result.summary_df.sort_values('VIF', ascending=True)
    
    color_map = {'OK': '#1D9E75', 'MODERATE': '#BA7517', 'HIGH': '#E24B4A'}
    colors = df['flag'].map(color_map)
    
    fig, ax = plt.subplots(figsize=(8, max(3, len(df) * 0.5)))
    
    bars = ax.barh(df['feature'], df['VIF'], color=colors, height=0.55, alpha=0.85)
    
    # Umbral SR 11-7
    ax.axvline(x=5,  color='#BA7517', linestyle='--', linewidth=1, alpha=0.7, label='Moderate (5)')
    ax.axvline(x=10, color='#E24B4A', linestyle='--', linewidth=1, alpha=0.7, label='High (10)')
    
    # Anotaciones de valor
    for bar, val in zip(bars, df['VIF']):
        ax.text(val + 0.1, bar.get_y() + bar.get_height() / 2,
                f'{val:.2f}', va='center', fontsize=9)
    
    # Leyenda
    patches = [
        mpatches.Patch(color='#1D9E75', label='OK (VIF < 5)'),
        mpatches.Patch(color='#BA7517', label='Moderate (5–10)'),
        mpatches.Patch(color='#E24B4A', label='High (> 10)'),
    ]
    ax.legend(handles=patches, loc='lower right', fontsize=8)
    
    status_color = '#1D9E75' if result.status == 'PASS' else '#E24B4A'
    ax.set_title(f"{title}  |  Status: {result.status}",
                 fontsize=11, color=status_color, fontweight='bold')
    ax.set_xlabel('Variance Inflation Factor (VIF)', fontsize=9)
    ax.spines[['top', 'right']].set_visible(False)
    plt.tight_layout()
    plt.show()


plot_vif(result_vif_clean,     title="VIF — Modelo base (sin colinealidad)")
plot_vif(result_vif_collinear, title="VIF — Modelo con feature redundante")

### 3.4 Threshold alternativo para modelos con justificación

In [ ]:
# Algunos modelos legacy tienen justificaciones documentadas para
# aceptar VIF hasta 15. El threshold es configurable.
result_vif_relaxed = vif_check(
    df=df_features_collinear,
    feature_cols=FEATURES_COLLINEAR,
    threshold=15.0,   # threshold elevado con justificación documentada
)

print(f"Status con threshold=10 : {result_vif_collinear.status}")
print(f"Status con threshold=15 : {result_vif_relaxed.status}")
print()
print("Nota: Si se usa threshold > 10, documentar justificación en el MRM report.")

---
## 4. Exportar evidencia a Excel

Todo `ValidationResult` puede exportarse a Excel para trazabilidad. El archivo incluye una pestaña **Summary** con los resultados cuantitativos y una pestaña **Metadata** con el status, nombre del test y warnings — listo para adjuntar al reporte de validación.

In [ ]:
from datetime import datetime
import os

# Crear carpeta de evidencia si no existe
os.makedirs('validation_evidence', exist_ok=True)

# Timestamp para versionado
ts = datetime.now().strftime('%Y%m%d_%H%M')

# Exportar resultados de compare
compare_path = f'validation_evidence/compare_dryrun_{ts}.xlsx'
result_strict.to_excel(compare_path)
print(f"Evidencia comparación : {compare_path}")

# Exportar resultados de VIF
vif_path = f'validation_evidence/vif_check_{ts}.xlsx'
result_vif_collinear.to_excel(vif_path)
print(f"Evidencia VIF         : {vif_path}")

# Verificar contenido
print()
print("Pestañas en el archivo VIF:")
import openpyxl
wb = openpyxl.load_workbook(vif_path)
for sheet in wb.sheetnames:
    ws = wb[sheet]
    print(f"  {sheet}: {ws.max_row - 1} filas de datos")

---
## 5. Flujo completo de validación

Este bloque integra todas las funciones en un flujo típico de validación SR 11-7 para un modelo PD logístico. Copiar y adaptar para cada ejercicio de validación.

In [ ]:
# ═══════════════════════════════════════════════════════════════════
#  FLUJO COMPLETO — ADAPTAR PARA CADA VALIDACIÓN
# ═══════════════════════════════════════════════════════════════════

MODEL_ID      = 'PD-RETAIL-V2.3'
VALIDATOR     = 'Team Quant Validation'
REFERENCE_DT  = '2025-Q4'
KEY_COLS      = ['obligor_id']
FEATURE_COLS  = ['ltv', 'dti', 'credit_age_yrs', 'utilization_rate']
VIF_THRESHOLD = 10.0
NUMERIC_TOL   = 1e-4    # tolerancia acordada con Model Owner

# ── 1. Cargar datos (reemplazar con paths reales) ───────────────────
# df_mo  = pd.read_csv('model_owner_output.csv')
# df_val = pd.read_csv('validator_dryrun.csv')
# df_feats = pd.read_csv('model_features.csv')

# Para este demo usamos los DataFrames ya creados:
df_mo    = df_model_owner
df_val   = df_dry_run
df_feats = df_features_collinear   # intencionalmente con problema

print(f"Modelo          : {MODEL_ID}")
print(f"Período         : {REFERENCE_DT}")
print(f"Validador       : {VALIDATOR}")
print(f"Fecha ejecución : {datetime.now().strftime('%Y-%m-%d %H:%M')}")
print("─" * 50)

In [ ]:
# ── 2. Test 1: Dry Run vs Model Owner ──────────────────────────────
print("TEST 1 — Benchmarking (Dry Run vs Model Owner)")
print("─" * 50)

r_compare = compare_dataframes(
    df_reference=df_mo,
    df_challenger=df_val,
    key_cols=KEY_COLS,
    numeric_tol=NUMERIC_TOL,
    label_reference=f'MO {MODEL_ID}',
    label_challenger='Validator Dry Run',
)

status_symbol = '✓' if r_compare.status == 'PASS' else '✗'
print(f"  {status_symbol} compare_dataframes : {r_compare.status}")
if r_compare.warnings:
    for w in r_compare.warnings:
        print(f"      ⚠ {w}")

max_diff = r_compare.summary_df['max_abs_diff'].max()
print(f"  Max diferencia absoluta : {max_diff:.2e}")
print()
r_compare.summary_df[['column', 'max_abs_diff', 'rows_exceeding_tol']]

In [ ]:
# ── 3. Test 2: Supuestos del modelo — VIF ──────────────────────────
print("TEST 2 — Supuestos: Multicolinealidad (VIF)")
print("─" * 50)

r_vif = vif_check(
    df=df_feats,
    feature_cols=FEATURES_COLLINEAR,
    threshold=VIF_THRESHOLD,
)

status_symbol = '✓' if r_vif.status == 'PASS' else '✗'
print(f"  {status_symbol} vif_check : {r_vif.status}")
if r_vif.warnings:
    for w in r_vif.warnings:
        print(f"      ⚠ {w}")
print()
r_vif.summary_df.sort_values('VIF', ascending=False)

In [ ]:
# ── 4. Resumen ejecutivo y exportación ─────────────────────────────
results = {
    'Benchmarking (Dry Run)': r_compare,
    'Multicolinealidad (VIF)': r_vif,
}

print(f"{'─'*50}")
print(f"RESUMEN EJECUTIVO — {MODEL_ID} | {REFERENCE_DT}")
print(f"{'─'*50}")
print(f"{'Test':<30} {'Status':>8}  {'Warnings':>8}")
print(f"{'─'*50}")

all_pass = True
for test_name, res in results.items():
    symbol = '✓' if res.status == 'PASS' else '✗'
    n_warn = len(res.warnings)
    print(f"  {symbol} {test_name:<28} {res.status:>6}  {n_warn:>6} warnings")
    if res.status != 'PASS':
        all_pass = False

print(f"{'─'*50}")
overall = 'PASS' if all_pass else 'FAIL — revisar tests en rojo'
print(f"  OVERALL: {overall}")
print()

# Exportar toda la evidencia
ts = datetime.now().strftime('%Y%m%d_%H%M')
for test_name, res in results.items():
    slug = test_name.lower().replace(' ', '_').replace('(', '').replace(')', '')
    path = f'validation_evidence/{MODEL_ID}_{slug}_{ts}.xlsx'
    res.to_excel(path)
    print(f"  Evidencia exportada → {path}")

---
## Referencia rápida

| Función | Cuándo usar | Output clave |
|---|---|---|
| `compare_dataframes(ref, cha, key_cols)` | Dry run vs Model Owner | `status`, `summary_df`, `warnings` |
| `vif_check(df, feature_cols, threshold)` | Test de colinealidad | `status`, `summary_df[flag]`, `details[flagged_features]` |
| `result.to_excel(path)` | Evidencia para auditoría | Archivo .xlsx con Summary + Metadata |

**Próximas funciones disponibles:**
- `psi_check()` — Population Stability Index (estabilidad de inputs)
- `csi_check()` — Characteristic Stability Index (estabilidad por segmento)  
- `backtesting_report()` — Gini, AUC-ROC, KS, calibración PD

---
*Generado con `validation-suite` 